# Glyph — Evaluation (этап 9) на GPU

Запускает полный evaluation-пайплайн на GPU Colab/Kaggle за ~15 минут вместо ~2-3 часов на CPU.

**Перед запуском:** `Runtime → Change runtime type → T4 GPU`.

## Что делает
1. Клонирует репозиторий Glyph
2. Устанавливает зависимости (torch с CUDA уже в Colab)
3. Читает retrieval-кэш из репозитория (без Qdrant)
4. Считает perplexity для base model и всех 4 адаптеров × 3 авторов
5. Генерирует ответы для 5 условий (baseline / rag_only / rag_lora r=4/8/16)
6. Считает стилевые метрики (TTR, POS, hapax ratio) и semantic similarity
7. Упаковывает результаты в zip и даёт скачать

## 1. Проверка GPU

In [ ]:
!nvidia-smi | head -20
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Клонирование репозитория

In [ ]:
%cd /content
!rm -rf glyph
!git clone --depth 1 https://github.com/tearfu1/glyph.git
%cd /content/glyph/ml

## 3. Установка зависимостей

В Colab уже есть torch с CUDA, поэтому его не переустанавливаем.

In [ ]:
!pip install -q transformers==4.51.0 peft==0.18.1 sentence-transformers==2.7.0 \
    datasets==2.19.0 pydantic-settings==2.2.0 qdrant-client==1.9.0 \
    razdel==0.5.0 pymorphy3==2.0.2 pandas==2.2.0 scikit-learn==1.4.0

## 4. Проверка артефактов

Адаптеры, processed-тексты и retrieval-кэш должны быть в репозитории.

In [ ]:
import os
print('Адаптеры:')
!ls models/adapters/
print('\nProcessed тексты:')
!ls data/processed/
print('\nRetrieval-кэш:')
cache_path = 'data/retrieval_cache.json'
assert os.path.exists(cache_path), f'Нет {cache_path} — запусти локально --prepare-retrieval-cache и закоммить'
!ls -lh data/retrieval_cache.json

## 5. Запуск evaluation

`--use-cache` — читаем retrieval из JSON, не трогаем Qdrant.

Оценка времени на T4:
- Perplexity: ~5 мин (base + 4 адаптера × 3 автора, forward pass)
- Generation: ~10 мин (5 условий × 10 вопросов × 3 автора = 150 ответов, ~4 сек/ответ на GPU)
- Метрики: <1 мин

In [ ]:
!python -m scripts.run_evaluation --use-cache data/retrieval_cache.json --num-questions 10

## 6. Просмотр результатов

In [ ]:
import pandas as pd

print('=== Perplexity ===')
display(pd.read_csv('models/evaluation/perplexity.csv'))

print('\n=== Style metrics ===')
display(pd.read_csv('models/evaluation/style_metrics.csv'))

print('\n=== Semantic similarity ===')
display(pd.read_csv('models/evaluation/semantic_similarity.csv'))

## 7. Примеры сгенерированных ответов

In [ ]:
import json

with open('models/evaluation/generation_samples.jsonl', 'r', encoding='utf-8') as f:
    samples = [json.loads(line) for line in f]

# Берём первый вопрос каждого автора для сравнения условий
for author in ['dostoevsky', 'chekhov', 'bulgakov']:
    author_samples = [s for s in samples if s['author'] == author]
    if not author_samples:
        continue
    q = author_samples[0]['question']
    print(f'\n{"="*80}\nАвтор: {author}\nВопрос: {q}\n{"="*80}')
    for s in author_samples:
        if s['question'] == q:
            print(f'\n[{s["condition"]}]\n{s["answer"]}')

## 8. Скачать результаты

In [ ]:
!cd models && zip -r /content/glyph_evaluation_results.zip evaluation/
from google.colab import files
files.download('/content/glyph_evaluation_results.zip')